In [1]:
!pip uninstall -y transformers huggingface_hub accelerate langchain langchain-core bitsandbytes
!pip install -U transformers accelerate huggingface_hub pypdf langchain langchain-core "bitsandbytes>=0.46.1"

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: huggingface_hub 1.11.0
Uninstalling huggingface_hub-1.11.0:
  Successfully uninstalled huggingface_hub-1.11.0
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: langchain 1.2.15
Uninstalling langchain-1.2.15:
  Successfully uninstalled langchain-1.2.15
Found existing installation: langchain-core 1.3.1
Uninstalling langchain-core-1.3.1:
  Successfully uninstalled langchain-core-1.3.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 101.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.9/136.9 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
import glob
import torch
import re
import gc
import pypdf
from typing import List
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate

In [3]:
pdf_files = glob.glob('/kaggle/input/**/*.pdf', recursive=True)
if not pdf_files:
    raise FileNotFoundError("No PDF file found in /kaggle/input/. Please check your upload.")
pdf_path = pdf_files[0]
print(f"Reading file: {pdf_path}\n")

model_name = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    quantization_config=quantization_config,
    device_map="auto"
)

Reading file: /kaggle/input/datasets/iabdallah/cvvvvv/CV.pdf



config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

...

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [4]:
def generate_text(prompt, max_length=2000, num_return_sequences=1):
    torch.cuda.empty_cache()
    gc.collect()
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_length = inputs.input_ids.shape[1]
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=800,
            num_return_sequences=num_return_sequences,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.3,
            pad_token_id=tokenizer.eos_token_id
        )
    
    return [tokenizer.decode(output[input_length:], skip_special_tokens=True) for output in outputs]

In [5]:
class EducationInfo(BaseModel):
    degree: str = Field(description="Degree obtained")
    institution: str = Field(description="Institution name")
    year: int = Field(description="Year of graduation")

class ExperienceInfo(BaseModel):
    role: str = Field(description="Job role or title")
    company: str = Field(description="Company name")
    years: str = Field(description="Years worked")

class ResumeSchema(BaseModel):
    full_name: str = Field(description="Full name of the candidate")
    email: str = Field(description="Email address of the candidate")
    education: List[EducationInfo] = Field(description="List of education details")
    skills: List[str] = Field(description="List of skills")
    experience: List[ExperienceInfo] = Field(description="List of work experience details")

output_parser = JsonOutputParser(pydantic_object=ResumeSchema)
format_instructions = output_parser.get_format_instructions()

In [6]:
cv_extraction_template = """
You are an HR assistant that extracts candidate profiles from resume snippets.
Extract the information and respond ONLY in valid JSON format.
{format_instructions}

Input:
"{user_input}"
"""

user_input = ""
with open(pdf_path, "rb") as file:
    reader = pypdf.PdfReader(file)
    for page in reader.pages:
        user_input += page.extract_text() + "\n\n"

user_input = user_input[:4000]

prompt = PromptTemplate(
    template=cv_extraction_template,
    input_variables=["user_input", "format_instructions"]
).format(user_input=user_input, format_instructions=format_instructions)

response = generate_text(prompt)[0]

def extract_json_block(text):
    pattern = r'```json\s*(.*?)\s*```'
    matches = re.findall(pattern, text, re.DOTALL)
    if matches:
        return matches[-1]
    
    pattern_fallback = r'\{.*\}'
    matches_fallback = re.findall(pattern_fallback, text, re.DOTALL)
    if matches_fallback:
        return matches_fallback[-1]
        
    return text

json_text = extract_json_block(response)
output_data = output_parser.parse(json_text)
print(output_data)


--- Final Output ---

{'full_name': 'ABDALLAH AHMED KHALIFA', 'email': 'abdallah.khalifa@proton.me', 'education': [{'degree': 'Bachelor of Science in Computer Science', 'institution': 'Faculty of Science, Alexandria University', 'year': None, 'expected_graduation': 'June 2027'}], 'skills': ['Generative AI: LLMs, RAG, Agentic AI, Prompt Engineering, LangChain, AI Application Development', 'Programming & Tools: Python, SQL, Git/GitHub, Jupyter Notebook, VS Code, Linux, Bash', 'Machine Learning & Deep Learning: Scikit-learn, TensorFlow, Keras, PyTorch, NumPy, Pandas, OpenCV, Neural Networks, CNNs, RNNs/LSTMs, Model Evaluation & Optimization', 'MLOps: MLflow, Flask, FastAPI, Model Deployment, Version Control, Reproducibility, Environment Management', 'Soft Skills: Problem Solving, Analytical Thinking, Communication, Teamwork, Adaptability, Time Management'], 'experience': []}
